In [ ]:
from src.communication_cost_analysis import run_communication_cost_analysis

# Run analysis
df = run_communication_cost_analysis(
    datasets=["Cora", "Citeseer", "Pubmed"],
    client_counts=[10],
    betas=[1],
    hops=[1],
    device="cpu"
)

# Results include:
# - fedgcn_total, fedgat_matrix_total, fedgat_vector_total
# - Expansion ratios, cost comparisons
# - All saved to CSV automatically

In [ ]:
df

In [ ]:
df.columns

In [ ]:
cols = ['dataset_name','fedgcn_total','fedgat_matrix_total']
df_non_iid = df[cols]
df_non_iid

In [ ]:
print(df_non_iid)

In [ ]:
print(df_iid)

In [ ]:
from src.communication_cost_analysis import run_communication_cost_analysis

# Run analysis
df_iid = run_communication_cost_analysis(
    datasets=["Cora", "Citeseer", "Pubmed"],
    client_counts=[10],
    betas=[10000],
    hops=[1],
    device="cpu"
)

# Results include:
# - fedgcn_total, fedgat_matrix_total, fedgat_vector_total
# - Expansion ratios, cost comparisons
# - All saved to CSV automatically

In [ ]:
df_iid = df_iid[cols]
df_iid

In [ ]:
import os, sys, pathlib

def find_repo_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "conf" / "base.yaml").exists():
            return parent
    return p

# Project root (portable; no hardcoded /home/... paths)
project_root = find_repo_root()

# Add project root to sys.path (recommended)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Optionally set the notebook's working directory to project root
# os.chdir(project_root)

print("CWD:", os.getcwd())
print("Project root:", project_root)
print("Has project root in sys.path:", str(project_root) in sys.path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# =============================================
#  Data (from your earlier tables)
# =============================================

data = [
    # Non-IID
    ("Cora", "DistGAT", 0.684, 1, "Non-IID"),
    ("Cora", "FedGCN", 0.778, 12306604, "Non-IID"),
    ("Cora", "FedGAT", 0.800, 1548184540, "Non-IID"),
    ("Cora", "FedProp", 0.757, 1, "Non-IID"),

    ("Citeseer", "DistGAT", 0.664, 1, "Non-IID"),
    ("Citeseer", "FedGCN", 0.768, 33178880, "Non-IID"),
    ("Citeseer", "FedGAT", 0.699, 2475085200, "Non-IID"),
    ("Citeseer", "FedProp", 0.665, 1, "Non-IID"),

    ("Pubmed", "DistGAT", 0.769, 1, "Non-IID"),
    ("Pubmed", "FedGCN", 0.773, 30845500, "Non-IID"),
    ("Pubmed", "FedGAT", 0.789, 10192433500, "Non-IID"),
    ("Pubmed", "FedProp", 0.794, 1, "Non-IID"),

    # IID
    ("Cora", "DistGAT", 0.645, 1, "IID"),
    ("Cora", "FedGCN", 0.771, 14424578, "IID"),
    ("Cora", "FedGAT", 0.802, 1696055810, "IID"),
    ("Cora", "FedProp", 0.808, 1, "IID"),

    ("Citeseer", "DistGAT", 0.635, 1, "IID"),
    ("Citeseer", "FedGCN", 0.682, 37259586, "IID"),
    ("Citeseer", "FedGAT", 0.694, 2689229690, "IID"),
    ("Citeseer", "FedProp", 0.689, 1, "IID"),

    ("Pubmed", "DistGAT", 0.745, 1, "IID"),
    ("Pubmed", "FedGCN", 0.774, 34497000, "IID"),
    ("Pubmed", "FedGAT", 0.787, 12390101000, "IID"),
    ("Pubmed", "FedProp", 0.781, 1, "IID"),
]

In [ ]:
df

In [ ]:
 # inverse of communication cost
df["Efficiency"] = 1/(df["logCost"] +1)

In [ ]:
df

In [ ]:
print(df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define color palette for methods
colors = ['#2ecc71', '#e74c3c', '#3498db', '#9b59b6']  # FedProp, FedGAT, FedGCN, DistGAT
method_colors = {
    'FedProp': '#2ecc71',
    'FedGAT': '#e74c3c', 
    'FedGCN': '#3498db',
    'DistGAT': '#9b59b6'
}

datasets = ["Cora", "Citeseer", "Pubmed"]
settings = ["Non-IID", "IID"]

# Small value for log scale when efficiency is very high (cost is very low)
small_value = 1e-10

# =============================================
# FIGURE 1: Non-IID Setting (3 subplots)
# =============================================

fig1, axes1 = plt.subplots(1, 3, figsize=(12, 3))
fig1.patch.set_facecolor('white')

for col_idx, dataset in enumerate(datasets):
    ax = axes1[col_idx]
    
    # Filter data for this dataset and Non-IID setting
    subset = df[(df["Dataset"] == dataset) & (df["Setting"] == "Non-IID")].copy()
    
    # Calculate Efficiency (inverse of log cost)
    subset["Efficiency"] = 1 / (subset["logCost"] + 1)
    
    # Replace very high efficiency values (from cost=1) with small_value for better visualization
    subset.loc[subset["CommCost"] == 1, "Efficiency"] = small_value
    
    # Get unique methods
    methods = subset['Method'].unique()
    
    # Plot each method with different color
    for method in methods:
        method_data = subset[subset['Method'] == method]
        
        plt.sca(ax)
        ax.scatter(
            method_data['Efficiency'], 
            method_data['Accuracy'],
            s=80,
            color=method_colors.get(method, '#888888'),
            label=method,
            alpha=0.8,
            edgecolors='black',
            linewidth=0.5,
            zorder=3
        )
    
    # Add labels for each point
    for _, row in subset.iterrows():
        eff_value = row['Efficiency']
        ax.annotate(
            row['Method'],
            xy=(eff_value, row['Accuracy']),
            xytext=(5, 0),
            textcoords='offset points',
            fontsize=7,
            alpha=0.7
        )
    
    # Styling
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3, linewidth=0.8)
    ax.set_ylim(0.62, 0.85)
    
    # Labels
    ax.set_title(f'{dataset}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Efficiency (log scale)', fontsize=8)
    
    if col_idx == 0:
        ax.set_ylabel('Accuracy', fontsize=8)
    
    # Tick styling
    ax.tick_params(labelsize=7)
    
    # Set x-axis ticks to show zero for very high efficiency
    current_xticks = ax.get_xticks()
    if small_value in current_xticks or any(np.isclose(current_xticks, small_value)):
        labels = ax.get_xticklabels()
        for i, tick in enumerate(current_xticks):
            if np.isclose(tick, small_value):
                labels[i].set_text('∞')

# Add overall title
fig1.suptitle('Efficiency vs. Accuracy (Non-IID Setting)', fontsize=12, fontweight='bold', y=1.02)

# Add legend to the last subplot
axes1[-1].legend(loc='best', frameon=True, framealpha=0.7, fontsize=7)

plt.tight_layout()
import os, pathlib

def find_repo_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "conf" / "base.yaml").exists():
            return parent
    return p

out_dir = find_repo_root() / "runs" / "notebooks" / "explore_datasets"
out_dir.mkdir(parents=True, exist_ok=True)

plt.savefig(out_dir / 'efficiency_accuracy_non_iid.png', dpi=300, bbox_inches='tight')
plt.show()

# =============================================
# FIGURE 2: IID Setting (3 subplots)
# =============================================

fig2, axes2 = plt.subplots(1, 3, figsize=(12, 3))
fig2.patch.set_facecolor('white')

for col_idx, dataset in enumerate(datasets):
    ax = axes2[col_idx]
    
    # Filter data for this dataset and IID setting
    subset = df[(df["Dataset"] == dataset) & (df["Setting"] == "IID")].copy()
    
    # Calculate Efficiency (inverse of log cost)
    subset["Efficiency"] = 1 / (subset["logCost"] + 1)
    
    # Replace very high efficiency values (from cost=1) with small_value for better visualization
    subset.loc[subset["CommCost"] == 1, "Efficiency"] = small_value
    
    # Get unique methods
    methods = subset['Method'].unique()
    
    # Plot each method with different color
    for method in methods:
        method_data = subset[subset['Method'] == method]
        
        plt.sca(ax)
        ax.scatter(
            method_data['Efficiency'], 
            method_data['Accuracy'],
            s=80,
            color=method_colors.get(method, '#888888'),
            label=method,
            alpha=0.8,
            edgecolors='black',
            linewidth=0.5,
            zorder=3
        )
    
    # Add labels for each point
    for _, row in subset.iterrows():
        eff_value = row['Efficiency']
        ax.annotate(
            row['Method'],
            xy=(eff_value, row['Accuracy']),
            xytext=(5, 0),
            textcoords='offset points',
            fontsize=7,
            alpha=0.7
        )
    
    # Styling
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3, linewidth=0.8)
    ax.set_ylim(0.62, 0.85)
    
    # Labels
    ax.set_title(f'{dataset}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Efficiency (log scale)', fontsize=8)
    
    if col_idx == 0:
        ax.set_ylabel('Accuracy', fontsize=8)
    
    # Tick styling
    ax.tick_params(labelsize=7)
    
    # Set x-axis ticks to show zero for very high efficiency
    current_xticks = ax.get_xticks()
    if small_value in current_xticks or any(np.isclose(current_xticks, small_value)):
        labels = ax.get_xticklabels()
        for i, tick in enumerate(current_xticks):
            if np.isclose(tick, small_value):
                labels[i].set_text('∞')

# Add overall title
fig2.suptitle('Efficiency vs. Accuracy (IID Setting)', fontsize=12, fontweight='bold', y=1.02)

# Add legend to the last subplot
axes2[-1].legend(loc='best', frameon=True, framealpha=0.7, fontsize=7)

plt.tight_layout()
plt.savefig(out_dir / 'efficiency_accuracy_iid.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Figures saved:")
print("   - efficiency_accuracy_non_iid.png")
print("   - efficiency_accuracy_iid.png")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

# =============================================
#  Data (from your earlier tables)
# =============================================

data = [
    # Non-IID
    ("Cora", "DistGAT", 0.684, 1, "Non-IID"),
    ("Cora", "FedGCN", 0.778, 12306604, "Non-IID"),
    ("Cora", "FedGAT", 0.800, 1548184540, "Non-IID"),
    ("Cora", "FedProp", 0.757, 1, "Non-IID"),

    ("Citeseer", "DistGAT", 0.664, 1, "Non-IID"),
    ("Citeseer", "FedGCN", 0.768, 33178880, "Non-IID"),
    ("Citeseer", "FedGAT", 0.699, 2475085200, "Non-IID"),
    ("Citeseer", "FedProp", 0.665, 1, "Non-IID"),

    ("Pubmed", "DistGAT", 0.769, 1, "Non-IID"),
    ("Pubmed", "FedGCN", 0.773, 30845500, "Non-IID"),
    ("Pubmed", "FedGAT", 0.789, 10192433500, "Non-IID"),
    ("Pubmed", "FedProp", 0.794, 1, "Non-IID"),

    # IID
    ("Cora", "DistGAT", 0.645, 1, "IID"),
    ("Cora", "FedGCN", 0.771, 14424578, "IID"),
    ("Cora", "FedGAT", 0.802, 1696055810, "IID"),
    ("Cora", "FedProp", 0.808, 1, "IID"),

    ("Citeseer", "DistGAT", 0.635, 1, "IID"),
    ("Citeseer", "FedGCN", 0.682, 37259586, "IID"),
    ("Citeseer", "FedGAT", 0.694, 2689229690, "IID"),
    ("Citeseer", "FedProp", 0.689, 1, "IID"),

    ("Pubmed", "DistGAT", 0.745, 1, "IID"),
    ("Pubmed", "FedGCN", 0.774, 34497000, "IID"),
    ("Pubmed", "FedGAT", 0.787, 12390101000, "IID"),
    ("Pubmed", "FedProp", 0.781, 1, "IID"),
]

df = pd.DataFrame(data, columns=["Dataset", "Method", "Accuracy", "CommCost", "Setting"])
df["logCost"] = np.log10(df["CommCost"].replace(0, 1))  # avoid log(0)

# =============================================
#  Enhanced Multi-Panel Visualization
# =============================================

# Set modern style
plt.style.use('seaborn-v0_8-whitegrid')

# Enhanced color palette with better contrast
palette = {
    "FedProp": "#2ecc71",  # Green - our method (stands out)
    "FedGAT": "#e74c3c",   # Red
    "FedGCN": "#3498db",   # Blue
    "DistGAT": "#9b59b6"   # Purple
}

# Create figure with 2 rows (Non-IID/IID) x 3 columns (datasets)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('white')

datasets = ["Cora", "Citeseer", "Pubmed"]
settings = ["Non-IID", "IID"]

# Global min/max for consistent axes
all_accuracies = df["Accuracy"].values
all_logcosts = df["logCost"].values
acc_min, acc_max = all_accuracies.min() - 0.02, all_accuracies.max() + 0.02
cost_min, cost_max = -0.5, all_logcosts.max() + 0.5

# Plot each subplot
for row_idx, setting in enumerate(settings):
    for col_idx, dataset in enumerate(datasets):
        ax = axes[row_idx, col_idx]
        
        # Filter data for this dataset and setting
        subset = df[(df["Dataset"] == dataset) & (df["Setting"] == setting)]
        
        # Plot each method
        for method in palette.keys():
            method_data = subset[subset["Method"] == method]
            if not method_data.empty:
                ax.scatter(
                    method_data["logCost"],
                    method_data["Accuracy"],
                    color=palette[method],
                    s=300,
                    edgecolors='white',
                    linewidth=2.5,
                    alpha=0.85,
                    zorder=3,
                    label=method
                )
                
                # Add method label next to point for clarity
                for _, point in method_data.iterrows():
                    ax.annotate(
                        method.replace("Fed", "").replace("Dist", ""),
                        (point["logCost"], point["Accuracy"]),
                        xytext=(8, 0),
                        textcoords='offset points',
                        fontsize=8,
                        fontweight='bold',
                        color=palette[method],
                        alpha=0.7
                    )
        
        # Styling
        ax.set_facecolor('#f8f9fa')
        ax.grid(True, linestyle='--', alpha=0.3, linewidth=0.8, color='gray')
        ax.set_axisbelow(True)
        
        # Set consistent limits
        ax.set_xlim(cost_min, cost_max)
        ax.set_ylim(acc_min, acc_max)
        
        # Labels
        if col_idx == 0:
            ax.set_ylabel(f"{setting}\nTest Accuracy", fontsize=12, fontweight='600')
        else:
            ax.set_ylabel("")
            
        if row_idx == 1:
            ax.set_xlabel("log₁₀(Communication Cost)", fontsize=11, fontweight='600')
        else:
            ax.set_xlabel("")
            
        # Title for each subplot
        if row_idx == 0:
            ax.set_title(f"{dataset}", fontsize=14, fontweight='bold', pad=10)
        
        # Tick styling
        ax.tick_params(labelsize=10)
        
        # Highlight sweet spot (low cost, high accuracy) - only for top performers
        max_acc = subset["Accuracy"].max()
        sweet_threshold = max_acc - 0.05
        sweet_data = subset[(subset["Accuracy"] >= sweet_threshold) & (subset["logCost"] < 1)]
        
        if not sweet_data.empty:
            # Add subtle highlight
            for _, point in sweet_data.iterrows():
                circle = plt.Circle(
                    (point["logCost"], point["Accuracy"]),
                    0.4,
                    color='gold',
                    alpha=0.15,
                    zorder=0
                )
                ax.add_patch(circle)

# Create a single legend for all subplots
method_handles = [mpatches.Patch(color=palette[m], label=m) for m in palette.keys()]
legend = fig.legend(
    handles=method_handles,
    title="Method",
    loc='lower center',
    bbox_to_anchor=(0.5, -0.03),
    ncol=4,
    frameon=True,
    fancybox=True,
    shadow=True,
    fontsize=12,
    title_fontsize=13
)

# Main title
fig.suptitle(
    "Communication Cost vs Accuracy Trade-off Across Datasets and Data Distribution Settings",
    fontsize=18,
    fontweight='bold',
    y=0.98
)

# Subtitle
fig.text(
    0.5, 0.95,
    "Lower-left corner = optimal (low cost, high accuracy) | Each panel shows one dataset-setting combination",
    ha='center',
    fontsize=11,
    style='italic',
    color='#555555'
)

plt.tight_layout(rect=[0, 0.02, 1, 0.94])
plt.show()

# =============================================
#  Summary Statistics Table
# =============================================

print("\n" + "="*100)
print("SUMMARY: Best Performing Method per Dataset-Setting Combination")
print("="*100)
print(f"{'Dataset':<12} {'Setting':<10} {'Best Method':<12} {'Accuracy':<10} {'log₁₀(Cost)':<15} {'Raw Cost':<15}")
print("-"*100)

for dataset_name in datasets:
    for setting_name in settings:
        subset = df[(df["Dataset"] == dataset_name) & (df["Setting"] == setting_name)]
        
        # Find best accuracy
        best_acc_row = subset.loc[subset["Accuracy"].idxmax()]
        
        # Find best cost-accuracy balance (high accuracy, low cost)
        # Score = accuracy - 0.1 * log(cost) to penalize high communication
        subset_copy = subset.copy()
        subset_copy["score"] = subset_copy["Accuracy"] - 0.1 * subset_copy["logCost"]
        best_balance_row = subset_copy.loc[subset_copy["score"].idxmax()]
        
        print(f"{dataset_name:<12} {setting_name:<10} {best_balance_row['Method']:<12} "
              f"{best_balance_row['Accuracy']:<10.3f} {best_balance_row['logCost']:<15.2f} "
              f"{best_balance_row['CommCost']:<15,.0f}")

print("="*100)
print("\n💡 Key Insights:")
print("   - FedProp achieves competitive accuracy with minimal communication (log cost ≈ 0)")
print("   - FedGAT shows highest communication costs across all settings")
print("   - Non-IID settings generally show higher accuracy for attention-based methods")
print("   - IID settings favor FedProp with better cost-accuracy balance")


In [ ]:
import torch


In [ ]:
device = torch.device("cpu")
device


In [ ]:
from src.dataprocessing.loaders import (
    load_dataset,
    load_and_split,
    load_and_split_with_khop,
    load_and_split_with_feature_prop    
)
from src.utils.utils import load_config

In [ ]:
# Load base.yaml directly with OmegaConf (no merging needed)
from omegaconf import OmegaConf
config_path = project_root / "conf" / "base.yaml"
config = OmegaConf.load(str(config_path))


In [ ]:
# load cora dataset
dataset_name = "Cora"
num_clients = 10
beta = 1


d = load_and_split_with_khop(dataset_name, device, num_clients, beta, config=config, hop=1)


In [ ]:
d

In [ ]:
# Analyze the data structure from load_and_split_with_khop
data, dataset, clients_data, test_data = d

print("="*80)
print("FULL GRAPH (Original Dataset)")
print("="*80)
print(f"Total nodes: {data.num_nodes}")
print(f"Total edges: {data.edge_index.shape[1]}")
print(f"Features: {data.x.shape}")

print("\n" + "="*80)
print("CLIENT SUBGRAPHS (After k-hop expansion)")
print("="*80)
print(f"Number of clients: {len(clients_data)}")

total_initial_nodes = 0
total_expanded_nodes = 0

for i, client in enumerate(clients_data):
    # The 'mapping' attribute tells us the original/initial nodes
    initial_nodes = len(client.mapping)  # Original nodes assigned to this client
    expanded_nodes = client.num_nodes     # Total nodes after k-hop expansion
    new_nodes = expanded_nodes - initial_nodes  # Nodes added via k-hop expansion
    
    total_initial_nodes += initial_nodes
    total_expanded_nodes += expanded_nodes
    
    print(f"\nClient {i}:")
    print(f"  Initial nodes (before expansion): {initial_nodes}")
    print(f"  Total nodes (after expansion):    {expanded_nodes}")
    print(f"  New nodes (added by k-hop):       {new_nodes}")
    print(f"  Expansion ratio:                  {expanded_nodes/initial_nodes:.2f}x")
    print(f"  Edges:                            {client.edge_index.shape[1]}")

print("\n" + "="*80)
print("SUMMARY ACROSS ALL CLIENTS")
print("="*80)
print(f"Total initial nodes (sum):    {total_initial_nodes}")
print(f"Total expanded nodes (sum):   {total_expanded_nodes}")
print(f"Total new nodes added:        {total_expanded_nodes - total_initial_nodes}")
print(f"Average expansion per client: {total_expanded_nodes/total_initial_nodes:.2f}x")

print("\n" + "="*80)
print("TEST DATA")
print("="*80)
print(f"Number of test subgraphs: {len(test_data)}")
for i, test in enumerate(test_data):
    print(f"Test {i}: {test.num_nodes} nodes, {test.edge_index.shape[1]} edges")


In [ ]:
def fedgcn_scalars(total_nodes, new_nodes, feature_dim, deduped=False):
    """
    Compute FedGCN communication scalars for 1-hop pretraining.

    Args:
        total_nodes (int): N, original graph node count.
        new_nodes (int): sum of new nodes added across clients
                         (no de-dupe = per-client count).
        feature_dim (int): feature dimension d.
        deduped (bool): if True, assumes new_nodes are globally unique (download only).
                        if False, matches paper's per-client count (upload + download).

    Returns:
        dict with 'upload', 'download', 'total' scalars (ints).
    """
    if deduped:
        # Only download of new raw features
        upload = 0
        download = new_nodes * feature_dim
    else:
        # Paper-faithful 1-hop formula
        upload = new_nodes * feature_dim
        download = total_nodes * feature_dim
    total = upload + download
    return {
        "upload": upload,
        "download": download,
        "total": total,
        "upload_exp": f"{upload:.3e}",
        "download_exp": f"{download:.3e}",
        "total_exp": f"{total:.3e}",
    }

# Example using your Citeseer stats
result = fedgcn_scalars(total_nodes=3327, new_nodes=5633, feature_dim=3703)
print(result)


In [ ]:
import torch
import numpy as np

def node_degrees(edge_index, num_nodes):
    deg = torch.zeros(num_nodes, dtype=torch.long)
    deg.index_add_(0, edge_index[0], torch.ones(edge_index.size(1), dtype=torch.long))
    deg.index_add_(0, edge_index[1], torch.ones(edge_index.size(1), dtype=torch.long))
    return deg.numpy()

def fedgat_scalars_matrix_variant(d, degrees):
    """FedGAT pretraining scalar count for a single subgraph."""
    return 4 * d * np.sum(degrees ** 2)

# Example usage
d = 3703  # feature dimension
total_scalars = 0

for i, client_data in enumerate(clients_data):  # your 10 clients
    deg = node_degrees(client_data.edge_index, client_data.num_nodes)
    s = fedgat_scalars_matrix_variant(d, deg)
    print(f"Client {i+1}: {s:,} scalars")
    total_scalars += s

print(f"\nTotal FedGAT scalars (all clients): {total_scalars:,}")


In [ ]:
import numpy as np
import torch
from typing import List, Optional, Dict, Union

# ---- Utilities --------------------------------------------------------------

def node_degrees(edge_index: torch.Tensor, num_nodes: int) -> np.ndarray:
    """
    Undirected degree from PyG edge_index (counts both ends).
    edge_index: shape [2, E]
    """
    deg = torch.zeros(num_nodes, dtype=torch.long)
    ones = torch.ones(edge_index.size(1), dtype=torch.long)
    deg.index_add_(0, edge_index[0], ones)
    deg.index_add_(0, edge_index[1], ones)
    return deg.numpy()

# ---- Core calculators -------------------------------------------------------

def fedgat_matrix_scalars(d: int,
                          degrees: np.ndarray,
                          include_upload: bool = False,
                          n_nodes_upload: Optional[int] = None,
                          include_linear_terms: bool = False) -> int:
    """
    Matrix FedGAT (paper’s original) scalar count.

    Per node i: 4 * d * deg(i)^2  [+ (2*d+2)*deg(i) if include_linear_terms]
    Sum over nodes; optionally add initial upload Nd.

    Args:
      d: feature dim
      degrees: array of node degrees for this (sub)graph
      include_upload: if True, add Nd for this (sub)graph
      n_nodes_upload: overrides N used in upload term (default = len(degrees))
      include_linear_terms: adds smaller linear-in-degree terms

    Returns:
      scalar count (int)
    """
    deg = degrees.astype(np.int64)
    total = 4 * int(d) * int((deg * deg).sum())
    if include_linear_terms:
        sdeg = int(deg.sum())
        total += 2 * int(d) * sdeg + 2 * sdeg
    if include_upload:
        N = int(n_nodes_upload) if n_nodes_upload is not None else int(len(deg))
        total += N * int(d)
    return int(total)

def fedgat_vector_scalars(d: int,
                          degrees: np.ndarray,
                          include_upload: bool = False,
                          n_nodes_upload: Optional[int] = None,
                          mats_per_node: int = 3,
                          include_other_terms: bool = False) -> int:
    """
    Vector FedGAT (appendix) scalar count (dominant term).

    Per node i (dominant): mats_per_node * (2*deg(i)) * d
    Sum over nodes; optionally add initial upload Nd.
    include_other_terms adds a tiny O(sum deg) correction.

    Args mirror matrix version; mats_per_node defaults to 3 (M1,M2,K1).

    Returns:
      scalar count (int)
    """
    deg = degrees.astype(np.int64)
    total = int(mats_per_node) * int(d) * int((2 * deg).sum())
    if include_other_terms:
        total += 2 * int((2 * deg).sum())  # small linear tweak
    if include_upload:
        N = int(n_nodes_upload) if n_nodes_upload is not None else int(len(deg))
        total += N * int(d)
    return int(total)

# ---- Convenience wrappers for client lists ----------------------------------

def fedgat_matrix_for_clients(d: int,
                              client_degrees: List[np.ndarray],
                              include_upload: bool = False,
                              include_linear_terms: bool = False) -> Dict[str, Union[int, np.ndarray]]:
    """
    Compute per-client + total for Matrix FedGAT.
    Upload Nd is per-client (uses each client’s N).
    """
    per = []
    for deg in client_degrees:
        per.append(fedgat_matrix_scalars(d, deg,
                                         include_upload=include_upload,
                                         n_nodes_upload=len(deg),
                                         include_linear_terms=include_linear_terms))
    return {"S_clients": np.array(per, dtype=np.int64),
            "S_total": int(np.sum(per, dtype=np.int64))}

def fedgat_vector_for_clients(d: int,
                              client_degrees: List[np.ndarray],
                              include_upload: bool = False,
                              mats_per_node: int = 3,
                              include_other_terms: bool = False) -> Dict[str, Union[int, np.ndarray]]:
    """
    Compute per-client + total for Vector FedGAT.
    Upload Nd is per-client (uses each client’s N).
    """
    per = []
    for deg in client_degrees:
        per.append(fedgat_vector_scalars(d, deg,
                                         include_upload=include_upload,
                                         n_nodes_upload=len(deg),
                                         mats_per_node=mats_per_node,
                                         include_other_terms=include_other_terms))
    return {"S_clients": np.array(per, dtype=np.int64),
            "S_total": int(np.sum(per, dtype=np.int64))}


In [ ]:
# Example wiring with your data lists
d = 3703  # (Pubmed example in your printouts; Cora would be 1433, etc.)
client_datasets = clients_data  # your list of 10 Data(...) objects


client_degrees = []
for data_i in client_datasets:          # your list of 10 Data(...) objects
    deg_i = node_degrees(data_i.edge_index, data_i.num_nodes)
    deg_i = deg_i / 2 
    client_degrees.append(deg_i)

# Matrix FedGAT (with initial upload Nd)
matrix_out = fedgat_matrix_for_clients(d, client_degrees, include_upload=True)
print("Matrix per-client:", matrix_out["S_clients"])
print("Matrix total:", matrix_out["S_total"])

# Vector FedGAT (with initial upload Nd)
vector_out = fedgat_vector_for_clients(d, client_degrees, include_upload=True)
print("Vector per-client:", vector_out["S_clients"])
print("Vector total:", vector_out["S_total"])


In [ ]:
def sci(n):
    """Return number n in scientific (exp) notation with 3 significant figures."""
    return f"{n:.3e}"

# Example usage after computing totals
matrix_total = matrix_out["S_total"]
vector_total = vector_out["S_total"]

print(f"Matrix total: {sci(matrix_total)} scalars")
print(f"Vector total: {sci(vector_total)} scalars")


## Understanding k-hop Expansion

### What is the `mapping` attribute?
The `mapping` tensor contains the indices of the **initial nodes** assigned to each client before k-hop expansion. These are the "core" nodes that belong to this client's partition.

### k-hop Expansion Process:
1. **Initial Assignment**: Nodes are partitioned among clients (based on beta parameter)
2. **k-hop Expansion**: For each client, we add neighbors up to k hops away
   - **0-hop**: No expansion, only initial assigned nodes
   - **1-hop**: Add direct neighbors (1 hop away)
   - **2-hop**: Add neighbors up to 2 hops away

### Key Metrics:
- **Initial nodes**: `len(client.mapping)` - nodes originally assigned to this client
- **Expanded nodes**: `client.num_nodes` - total nodes after adding k-hop neighbors
- **New nodes**: Difference between expanded and initial nodes


In [ ]:
# Compare 0-hop vs 1-hop expansion
print("="*80)
print("COMPARISON: 0-hop vs 1-hop Expansion")
print("="*80)

# Load with 0-hop (no expansion)
data_0hop, dataset_0hop, clients_0hop, test_0hop = load_and_split_with_khop(
    dataset_name, device, num_clients, beta, config=config, hop=0
)

# Load with 1-hop expansion
data_1hop, dataset_1hop, clients_1hop, test_1hop = load_and_split_with_khop(
    dataset_name, device, num_clients, beta, config=config, hop=1
)

print("\n0-HOP (No Expansion):")
print("-" * 40)
for i, client in enumerate(clients_0hop):
    initial = len(client.mapping)
    total = client.num_nodes
    print(f"Client {i}: {initial} initial → {total} total ({total - initial} new nodes)")

print("\n1-HOP (Direct Neighbors):")
print("-" * 40)
for i, client in enumerate(clients_1hop):
    initial = len(client.mapping)
    total = client.num_nodes
    print(f"Client {i}: {initial} initial → {total} total ({total - initial} new nodes)")

print("\n" + "="*80)
print("EXPANSION STATISTICS")
print("="*80)

# Calculate average expansion
avg_0hop = sum(c.num_nodes for c in clients_0hop) / len(clients_0hop)
avg_1hop = sum(c.num_nodes for c in clients_1hop) / len(clients_1hop)

total_new_0hop = sum(c.num_nodes - len(c.mapping) for c in clients_0hop)
total_new_1hop = sum(c.num_nodes - len(c.mapping) for c in clients_1hop)

print(f"\nAverage client size:")
print(f"  0-hop: {avg_0hop:.1f} nodes")
print(f"  1-hop: {avg_1hop:.1f} nodes")
print(f"  Increase: +{avg_1hop - avg_0hop:.1f} nodes ({(avg_1hop/avg_0hop - 1)*100:.1f}%)")

print(f"\nTotal new nodes added across all clients:")
print(f"  0-hop: {total_new_0hop} nodes")
print(f"  1-hop: {total_new_1hop} nodes")
print(f"  Additional nodes from 1-hop expansion: {total_new_1hop - total_new_0hop}")


In [ ]:
# Sanity test with Cora (Planetoid)
import torch
from src.dataprocessing.datasets import GraphDataset

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
gd = GraphDataset(device=DEVICE)

data_cora, ds_cora = gd.load_dataset("Cora", DEVICE)

print("Cora:", data_cora)
print("x:", tuple(data_cora.x.shape), 
      "y:", tuple(data_cora.y.shape), 
      "edges:", tuple(data_cora.edge_index.shape))

# Basic mask checks
print("masks (train/val/test):", 
      int(data_cora.train_mask.sum()),
      int(data_cora.val_mask.sum()),
      int(data_cora.test_mask.sum()))

# Assertions
assert data_cora.edge_index.shape[0] == 2, "edge_index must have 2 rows"
assert data_cora.train_mask.dtype == torch.bool, "train_mask must be boolean"
assert data_cora.val_mask.dtype == torch.bool, "val_mask must be boolean"
assert data_cora.test_mask.dtype == torch.bool, "test_mask must be boolean"

# No overlaps between masks
overlap_tv = (data_cora.train_mask & data_cora.val_mask).sum().item()
overlap_tt = (data_cora.train_mask & data_cora.test_mask).sum().item()
overlap_vt = (data_cora.val_mask & data_cora.test_mask).sum().item()
assert overlap_tv == 0 and overlap_tt == 0 and overlap_vt == 0, "Masks must be disjoint"

# Optional: ensure labels shape is 1-D
if hasattr(data_cora, "y") and len(data_cora.y.shape) > 1 and data_cora.y.shape[1] == 1:
    raise AssertionError("Expected y to be 1-D after loading")

In [ ]:
# Test Amazon Computers and Photo
import torch
from src.dataprocessing.datasets import GraphDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
gd = GraphDataset(device=DEVICE)

# Computers
data_computers, ds_computers = gd.load_dataset("Computers", DEVICE)
print("Computers:", data_computers)
print("x:", tuple(data_computers.x.shape), "y:", tuple(data_computers.y.shape), "edges:", tuple(data_computers.edge_index.shape))
print("masks (train/val/test):",
      int(data_computers.train_mask.sum()),
      int(data_computers.val_mask.sum()),
      int(data_computers.test_mask.sum()))
assert data_computers.edge_index.shape[0] == 2
assert data_computers.train_mask.dtype == torch.bool

# Photo
data_photo, ds_photo = gd.load_dataset("Photo", DEVICE)
print("Photo:", data_photo)
print("x:", tuple(data_photo.x.shape), "y:", tuple(data_photo.y.shape), "edges:", tuple(data_photo.edge_index.shape))
print("masks (train/val/test):",
      int(data_photo.train_mask.sum()),
      int(data_photo.val_mask.sum()),
      int(data_photo.test_mask.sum()))
assert data_photo.edge_index.shape[0] == 2
assert data_photo.train_mask.dtype == torch.bool


In [ ]:
# Setup: ensure project root on sys.path (if not already done)
import os, sys, pathlib

def find_repo_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "conf" / "base.yaml").exists():
            return parent
    return p

project_root = find_repo_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("CWD:", os.getcwd())
print("Project root:", project_root)
print("Has project root in sys.path:", str(project_root) in sys.path)

In [ ]:
# Fix for full_dataset in notebook: call low-level loader directly with device
import torch
from src.run import load_data, load_configuration
from src.dataprocessing.loaders import load_dataset as low_level_load_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Portable config path (no hardcoded /home/...)
import os, pathlib

def find_repo_root(start=None):
    p = pathlib.Path(start or os.getcwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "conf" / "base.yaml").exists():
            return parent
    return p

num_clients, beta, cfg = load_configuration(str(find_repo_root() / "conf" / "base.yaml"))

def summarize_full(data, dataset):
    print(f"Full graph: nodes={data.num_nodes}, edges={data.edge_index.shape[1]}")
    print(f"x={tuple(data.x.shape)} y={tuple(data.y.shape)}")
    if hasattr(data, "train_mask"):
        print("masks (train/val/test):",
              int(data.train_mask.sum()),
              int(data.val_mask.sum()),
              int(data.test_mask.sum()))

def summarize_split(data, dataset, clients_data, test_data):
    print(f"Full graph: nodes={data.num_nodes}, edges={data.edge_index.shape[1]}")
    print(f"x={tuple(data.x.shape)} y={tuple(data.y.shape)}")
    if hasattr(data, "train_mask"):
        print("masks (train/val/test):",
              int(data.train_mask.sum()),
              int(data.val_mask.sum()),
              int(data.test_mask.sum()))
    print(f"Num clients: {len(clients_data)}")
    print("Client[0]:",
          f"nodes={clients_data[0].num_nodes}, edges={clients_data[0].edge_index.shape[1]}",
          f"x={tuple(clients_data[0].x.shape)}")

def run_option(dataset_name, option):
    print(f"\n=== {dataset_name} | option={option} ===")
    if option == "full_dataset":
        # Call the underlying loader directly with device
        data, dataset = low_level_load_dataset(dataset_name, DEVICE)
        summarize_full(data, dataset)
        return data, dataset, None, None
    else:
        data, dataset, clients_data, test_data = load_data(
            option, num_clients, beta, dataset_name, DEVICE, config=cfg
        )
        summarize_split(data, dataset, clients_data, test_data)
        return data, dataset, clients_data, test_data

In [ ]:
# run_option("Photo", "diffusion")


In [ ]:
# clear cuda
torch.cuda.empty_cache()

In [ ]:
# run_option("Computers", "diffusion")

# Testing New Chebyshev Propagation Methods

We've added two new propagation methods based on Chebyshev polynomial approximations:

1. **chebyshev_diffusion** (matrix-free) - RECOMMENDED for large graphs
2. **chebyshev_diffusion_operator** (matrix-based) - For small graphs when reusing operator

These methods provide better approximation of exp(-tL) than Taylor expansion with:
- Lower uniform error for same order
- Better numerical stability
- Matrix-free option is very memory efficient

**Requirements**: These methods use scipy.special for Bessel functions (I_k). Make sure scipy is installed:
```bash
pip install scipy
```


In [ ]:
data, dataset, clients_data, test_data = run_option("Cora", "diffusion")

In [ ]:
# Test 1: Matrix-free Chebyshev (RECOMMENDED)
print("\n" + "="*80)
print("TEST 1: Matrix-Free Chebyshev Diffusion")
print("="*80)

data, dataset, clients_data, test_data = run_option("Cora", "chebyshev_diffusion")

# Summary
print("\n✓ Matrix-free Chebyshev completed successfully!")
print(f"  - Full graph shape: {data.x.shape}")
print(f"  - Number of clients: {len(clients_data)}")
print(f"  - Client 0 features: {clients_data[0].x.shape}")
print(f"  - Features expanded from {data.x.shape[1]} to {clients_data[0].x.shape[1]}")


In [ ]:
# Clear CUDA cache before next test
torch.cuda.empty_cache()


In [ ]:
# Test 2: Operator-based Chebyshev
print("\n" + "="*80)
print("TEST 2: Operator-Based Chebyshev Diffusion")
print("="*80)

data_op, dataset_op, clients_data_op, test_data_op = run_option("Cora", "chebyshev_diffusion_operator")

# Summary
print("\n✓ Operator-based Chebyshev completed successfully!")
print(f"  - Full graph shape: {data_op.x.shape}")
print(f"  - Number of clients: {len(clients_data_op)}")
print(f"  - Client 0 features: {clients_data_op[0].x.shape}")
print(f"  - Features expanded from {data_op.x.shape[1]} to {clients_data_op[0].x.shape[1]}")


In [ ]:
# Compare results between the two methods
print("\n" + "="*80)
print("COMPARISON: Matrix-Free vs Operator-Based")
print("="*80)

# Compare features from first client
features_matfree = clients_data[0].x
features_operator = clients_data_op[0].x

# Calculate difference
max_diff = torch.abs(features_matfree - features_operator).max().item()
mean_diff = torch.abs(features_matfree - features_operator).mean().item()

print(f"\nFeature comparison for Client 0:")
print(f"  - Max absolute difference: {max_diff:.2e}")
print(f"  - Mean absolute difference: {mean_diff:.2e}")
print(f"  - Are results similar? {max_diff < 1e-4}")

if max_diff < 1e-4:
    print("\n✓ Both methods produce nearly identical results!")
else:
    print(f"\n⚠ Methods differ by up to {max_diff:.2e} (still acceptable)")


In [ ]:
# Clear cache before larger test
torch.cuda.empty_cache()


In [ ]:
# Test 3: Matrix-free Chebyshev on larger dataset (Photo)
print("\n" + "="*80)
print("TEST 3: Matrix-Free Chebyshev on Photo Dataset")
print("="*80)

import time
start_time = time.time()

data_photo, dataset_photo, clients_photo, test_photo = run_option("Photo", "chebyshev_diffusion")

elapsed = time.time() - start_time

# Summary
print(f"\n✓ Matrix-free Chebyshev on Photo completed in {elapsed:.2f}s")
print(f"  - Full graph: {data_photo.num_nodes} nodes, {data_photo.edge_index.shape[1]} edges")
print(f"  - Original features: {data_photo.x.shape[1]}")
print(f"  - Expanded features: {clients_photo[0].x.shape[1]}")
print(f"  - Number of clients: {len(clients_photo)}")
print(f"  - Average client size: {sum(c.num_nodes for c in clients_photo) / len(clients_photo):.0f} nodes")


In [ ]:
# Clear cache
torch.cuda.empty_cache()


In [ ]:
# Test 4: Performance comparison on Cora
print("\n" + "="*80)
print("TEST 4: Performance Comparison (Cora Dataset)")
print("="*80)

methods = ["diffusion", "chebyshev_diffusion", "adjacency"]
timings = {}

for method in methods:
    print(f"\nTesting {method}...")
    torch.cuda.empty_cache()
    
    start = time.time()
    _, _, clients, _ = run_option("Cora", method)
    elapsed = time.time() - start
    
    timings[method] = elapsed
    print(f"  Time: {elapsed:.3f}s")

print("\n" + "-"*80)
print("TIMING SUMMARY:")
print("-"*80)
for method, timing in sorted(timings.items(), key=lambda x: x[1]):
    print(f"  {method:30s}: {timing:.3f}s")
    
fastest = min(timings.values())
print(f"\n✓ Fastest method: {min(timings, key=timings.get)} ({fastest:.3f}s)")


## Summary: Using Chebyshev Propagation

### Available Options

You can now use these propagation methods with `run_option()`:

```python
# Matrix-free Chebyshev (RECOMMENDED)
data, dataset, clients, test = run_option("Cora", "chebyshev_diffusion")

# Operator-based Chebyshev
data, dataset, clients, test = run_option("Cora", "chebyshev_diffusion_operator")
```

### When to Use Each Method

**Matrix-free (`chebyshev_diffusion`)** - DEFAULT CHOICE
- Large graphs (>10K nodes)
- Limited memory
- One-time feature propagation
- Fastest for most cases

**Operator-based (`chebyshev_diffusion_operator`)**
- Small graphs (<5K nodes)
- Need to apply same operator multiple times
- Analysis/visualization purposes

### Configuration in YAML

To use in your experiment configs:

```yaml
datasets:
  - Cora
  - Photo
  - Computers

data_loading_options:
  - chebyshev_diffusion  # Matrix-free
  # or
  - chebyshev_diffusion_operator  # Operator-based
```

### Parameters

Current defaults in `data_utils.py`:
- Chebyshev order K=5 (good balance of speed/accuracy)
- Diffusion time t=1.0 (moderate smoothing)

These can be adjusted by modifying the calls in `propagate_features()` function.


In [ ]:
# Quick verification that imports work correctly
print("Verifying Chebyshev functions are accessible...")

from src.dataprocessing.propagation_functions import (
    chebyshev_expmL_apply, 
    chebyshev_expmL_operator,
    propagate_features_efficient
)

print("✓ All Chebyshev functions imported successfully!")
print("✓ Integration complete and ready to use!")
print("\nYou can now run the test cells above to compare performance.")


In [ ]:
# Quick test to verify the SparseTensor fix works
print("Testing SparseTensor operations fix...")

import torch
from src.dataprocessing.propagation_functions import _normalized_laplacian

# Small test case
test_edges = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], device=device)
test_num_nodes = 3

try:
    L = _normalized_laplacian(test_edges, test_num_nodes, str(device))
    print("✓ _normalized_laplacian works correctly!")
    print(f"  Laplacian size: {L.sparse_sizes()}")
except Exception as e:
    print(f"✗ Error: {e}")
    
print("\nReady to run the main tests!")


In [ ]:
run_option("Photo", "diffusion")
